In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/module1_spot_v0.parquet")
df.head()


,ts,open,high,low,close,volume,log_return,regime,split
0,2017-08-18 00:00:00+00:00,4285.08,4371.52,3938.77,4108.37,1199.888264,-0.042113,1,train
1,2017-08-19 00:00:00+00:00,4108.37,4184.69,3850.00,4139.98,381.309763,0.007665,2,train
2,2017-08-20 00:00:00+00:00,4120.98,4211.08,4032.62,4086.29,467.083022,-0.013053,1,train
3,2017-08-21 00:00:00+00:00,4069.13,4119.62,3911.79,4016.00,691.743060,-0.017351,1,train
4,2017-08-22 00:00:00+00:00,4016.00,4104.82,3400.00,4040.00,966.684858,0.005958,2,train


In [2]:
assert {'ts','log_return','regime','split'}.issubset(df.columns)


In [3]:
df['ret_5']  = df['log_return'].rolling(5).sum()
df['ret_10'] = df['log_return'].rolling(10).sum()
df['ret_20'] = df['log_return'].rolling(20).sum()
df['vol_10'] = df['log_return'].rolling(10).std()
df['vol_20'] = df['log_return'].rolling(20).std()


In [4]:
from numpy.polynomial.polynomial import polyfit

def slope(series):
    x = np.arange(len(series))
    b, _ = polyfit(x, series.values, 1)
    return b

df['trend_slope_20'] = (
    df['close']
    .rolling(20)
    .apply(slope, raw=False)
)


In [5]:
features = [
    'ret_5','ret_10','ret_20',
    'vol_10','vol_20',
    'trend_slope_20'
]

df_model = df.dropna(subset=features + ['regime']).reset_index(drop=True)


In [6]:
train_end = pd.Timestamp("2020-01-01", tz="UTC")
val_end   = pd.Timestamp("2023-01-01", tz="UTC")

def assign_split(ts):
    if ts < train_end:
        return "train"
    elif ts < val_end:
        return "val"
    else:
        return "test"

df_model["split"] = df["ts"].apply(assign_split)
df_model["split"].value_counts()


split
val      1096
test     1075
train     866
Name: count, dtype: int64

In [7]:
train = df_model[df_model['split']=='train']
val   = df_model[df_model['split']=='val']
test  = df_model[df_model['split']=='test']

X_train = train[features]
y_train = train['regime']

X_val = val[features]
y_val = val['regime']

X_test = test[features]
y_test = test['regime']


In [8]:
from sklearn.ensemble import GradientBoostingClassifier

clf = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

clf.fit(X_train, y_train)


,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``,

In [9]:
train_probs = clf.predict_proba(X_train)
val_probs   = clf.predict_proba(X_val)
test_probs  = clf.predict_proba(X_test)


In [10]:
y_vol_train = train['vol_20']
y_vol_val   = val['vol_20']
y_vol_test  = test['vol_20']


In [11]:
from sklearn.ensemble import RandomForestRegressor

vol_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

vol_model.fit(X_train, y_vol_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [12]:
vol_pred_train = vol_model.predict(X_train)
vol_pred_val   = vol_model.predict(X_val)
vol_pred_test  = vol_model.predict(X_test)


In [13]:
low_thr  = np.quantile(vol_pred_train, 0.33)
high_thr = np.quantile(vol_pred_train, 0.66)

def vol_regime(v):
    if v <= low_thr:
        return 'low'
    elif v <= high_thr:
        return 'mid'
    else:
        return 'high'


In [14]:
def trend_state(slope, eps=1e-5):
    if slope > eps:
        return 'up'
    elif slope < -eps:
        return 'down'
    else:
        return 'flat'


In [15]:
def build_state(df_part, probs, vol_pred):
    out = pd.DataFrame({
        'ts': df_part['ts'].values,
        'p0': probs[:,0],
        'p1': probs[:,1],
        'p2': probs[:,2],
        'p3': probs[:,3],
        'p4': probs[:,4],
        'vol_pred': vol_pred,
        'vol_regime': [vol_regime(v) for v in vol_pred],
        'trend_state': [
            trend_state(s) for s in df_part['trend_slope_20']
        ]
    })
    return out


In [16]:
module1_train = build_state(train, train_probs, vol_pred_train)
module1_val   = build_state(val, val_probs, vol_pred_val)
module1_test  = build_state(test, test_probs, vol_pred_test)


In [17]:
module1_train.count()

ts             866
p0             866
p1             866
p2             866
p3             866
p4             866
vol_pred       866
vol_regime     866
trend_state    866
dtype: int64

In [18]:
module1_train.to_parquet("data/module1_state_train.parquet", index=False)
module1_val.to_parquet("data/module1_state_val.parquet", index=False)
module1_test.to_parquet("data/module1_state_test.parquet", index=False)



In [19]:
import numpy as np
import pandas as pd

def compute_volatility(returns, span=20):
    return returns.ewm(span=span, adjust=False).std()

def get_triple_barrier_events(
    close,
    vol,
    pt_sl=(1.0, 1.0),
    max_holding=20
):
    """
    Generate triple-barrier events and labels.

    Returns:
        events_df with columns:
        - t0 : event start time
        - t1 : event end time
        - label : {-1, 0, +1}
        - ret : realized log-return
        - sigma : volatility at entry
    """

    records = []

    for i in range(len(close) - max_holding):
        t0 = close.index[i]
        p0 = close.iloc[i]
        sigma = vol.iloc[i]

        if np.isnan(sigma) or sigma <= 0:
            continue

        pt = pt_sl[0] * sigma
        sl = -pt_sl[1] * sigma

        t1 = close.index[i + max_holding]
        label = 0

        for j in range(1, max_holding + 1):
            ret = np.log(close.iloc[i + j] / p0)

            if ret >= pt:
                t1 = close.index[i + j]
                label = +1
                break

            if ret <= sl:
                t1 = close.index[i + j]
                label = -1
                break

        realized_ret = np.log(close.loc[t1] / p0)

        records.append({
            "t0": t0,
            "t1": t1,
            "label": label,
            "ret": realized_ret,
            "sigma": sigma
        })

    events = pd.DataFrame(records)
    events.index.name = "event_id"
    return events

# prerequisites
close = df["close"]
returns = np.log(close / close.shift(1)).dropna()
vol = compute_volatility(returns, span=20)

# align
idx = close.index.intersection(vol.index)
close = close.loc[idx]
vol = vol.loc[idx]

events = get_triple_barrier_events(
    close=close,
    vol=vol,
    pt_sl=(1.0, 1.0),
    max_holding=20
)

print(events.head())
print(events["label"].value_counts(normalize=True))


          t0  t1  label       ret     sigma
event_id                                   
0          2   3     -1 -0.017351  0.014650
1          3   5      1  0.024112  0.015898
2          4   5      1  0.018154  0.012974
3          5   6      1  0.047933  0.012611
4          6  11      1  0.060999  0.019708
label
 1    0.544496
-1    0.450560
 0    0.004944
Name: proportion, dtype: float64


In [20]:
def rolling_trend_slope(series, window=20):
    slopes = []
    for i in range(len(series)):
        if i < window:
            slopes.append(np.nan)
        else:
            y = series.iloc[i-window:i].values
            x = np.arange(window)
            slopes.append(np.polyfit(x, y, 1)[0])
    return pd.Series(slopes, index=series.index)

trend_slope_20 = rolling_trend_slope(close, window=20)


In [21]:
def generate_primary_signal(trend_slope, threshold=0.0):
    """
    +1  → long bias
    -1  → short bias
     0  → no signal
    """
    signal = np.sign(trend_slope)
    signal[np.abs(trend_slope) <= threshold] = 0
    return signal

primary_signal = generate_primary_signal(trend_slope_20)


In [22]:
events["primary_signal"] = primary_signal.reindex(events["t0"]).values
events = events.dropna(subset=["primary_signal"])


In [23]:
def compute_meta_label(events):
    """
    Meta-label:
    1 → trade was successful
    0 → trade was not successful / should not trade
    """
    meta = []

    for _, row in events.iterrows():
        sig = row["primary_signal"]
        lbl = row["label"]

        if sig == 0:
            meta.append(0)
        elif sig == lbl:
            meta.append(1)
        else:
            meta.append(0)

    return pd.Series(meta, index=events.index)

events["meta_label"] = compute_meta_label(events)
print(events["meta_label"].value_counts(normalize=True))


meta_label
1    0.518408
0    0.481592
Name: proportion, dtype: float64


In [24]:
events.head()

,t0,t1,label,ret,sigma,primary_signal,meta_label
event_id,,,,,,,
19,21,26,-1,-0.082237,0.051541,1.0,0
20,22,26,-1,-0.076619,0.048957,1.0,0
21,23,27,-1,-0.258653,0.047303,1.0,0
22,24,26,-1,-0.064729,0.045503,1.0,0
23,25,26,-1,-0.054039,0.043313,-1.0,1


In [25]:
events.tail()

,t0,t1,label,ret,sigma,primary_signal,meta_label
event_id,,,,,,,
3029,3031,3035,1,0.036802,0.027984,-1.0,0
3030,3032,3035,1,0.037846,0.026623,1.0,1
3031,3033,3041,-1,-0.044833,0.025750,1.0,0
3032,3034,3040,-1,-0.027542,0.024520,1.0,0
3033,3035,3038,-1,-0.026352,0.024336,1.0,0


In [26]:
events.columns


Index(['t0', 't1', 'label', 'ret', 'sigma', 'primary_signal', 'meta_label'], dtype='object')

In [27]:
import pandas as pd
import numpy as np

def get_concurrency(events):
    """
    events: DataFrame with t0, t1
    returns: Series indexed by time with concurrency count
    """
    timeline = []

    for _, row in events.iterrows():
        timeline.append((row['t0'], 1))
        timeline.append((row['t1'] + 1, -1))

    timeline = pd.DataFrame(timeline, columns=['time', 'delta'])
    timeline = timeline.sort_values('time')

    timeline['concurrency'] = timeline['delta'].cumsum()
    timeline = timeline.drop(columns='delta')

    return timeline.set_index('time')['concurrency']

concurrency_series = get_concurrency(events)
concurrency_series.head()


time
21.0    1
22.0    2
23.0    3
24.0    4
25.0    5
Name: concurrency, dtype: int64

In [28]:
def compute_event_uniqueness(events, concurrency):
    """
    Returns a Series of average uniqueness per event.
    """
    uniqueness = {}

    for idx, row in events.iterrows():
        event_times = concurrency.loc[row['t0']:row['t1']]
        uniqueness[idx] = (1.0 / event_times).mean()

    return pd.Series(uniqueness)

events['uniqueness'] = compute_event_uniqueness(
    events,
    concurrency_series
)


In [29]:
events['uniqueness'].describe()


count    3015.000000
mean        0.237938
std         0.107873
min         0.058905
25%         0.158383
50%         0.223341
75%         0.291667
max         0.750000
Name: uniqueness, dtype: float64

In [30]:
# Step 1: Sample weights
events['sample_weight'] = (
    events['uniqueness'] /
    events['uniqueness'].mean()
)

# Safety cap (recommended)
events['sample_weight'] = events['sample_weight'].clip(upper=5)

events['sample_weight'].describe()


count    3015.000000
mean        1.000000
std         0.453365
min         0.247566
25%         0.665649
50%         0.938655
75%         1.225812
max         3.152088
Name: sample_weight, dtype: float64

In [31]:
train_df = events[events['label'] != 0].copy()
X = train_df[['primary_signal']]   # keep minimal
y = train_df['label']
w = train_df['sample_weight']


In [32]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)


In [33]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000
)


In [34]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

precisions = []
recalls = []
f1s = []

for train_idx, test_idx in tscv.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    w_tr = w.iloc[train_idx]

    model.fit(X_tr, y_tr, sample_weight=w_tr)

    preds = model.predict(X_te)

    precisions.append(
        precision_score(y_te, preds, average="macro", zero_division=0)
    )
    recalls.append(
        recall_score(y_te, preds, average="macro", zero_division=0)
    )
    f1s.append(
        f1_score(y_te, preds, average="macro", zero_division=0)
    )


In [35]:
print({
    "precision": np.mean(precisions),
    "recall": np.mean(recalls),
    "f1": np.mean(f1s),
    # "signal_rate": 1.0  # by design at primary layer
})


{'precision': np.float64(0.47013662783676474), 'recall': np.float64(0.5142531461834015), 'f1': np.float64(0.4852132221107032)}


In [36]:
# Fit primary model once on all eligible events
model.fit(X, y, sample_weight=w)

# Directional prediction (±1)
train_df['primary_pred'] = model.predict(X)
train_df['primary_prob'] = model.predict_proba(X).max(axis=1)

train_df

,t0,t1,label,ret,sigma,primary_signal,meta_label,uniqueness,sample_weight,primary_pred,primary_prob
event_id,,,,,,,,,,,
19,21,26,-1,-0.082237,0.051541,1.0,0,0.408333,1.716137,1,0.549342
20,22,26,-1,-0.076619,0.048957,1.0,0,0.290000,1.218807,1,0.549342
21,23,27,-1,-0.258653,0.047303,1.0,0,0.285185,1.198572,1,0.549342
22,24,26,-1,-0.064729,0.045503,1.0,0,0.205556,0.863906,1,0.549342
23,25,26,-1,-0.054039,0.043313,-1.0,1,0.183333,0.770510,1,0.525531
...,...,...,...,...,...,...,...,...,...,...,...
3029,3031,3035,1,0.036802,0.027984,-1.0,0,0.235714,0.990656,1,0.525531
3030,3032,3035,1,0.037846,0.026623,1.0,1,0.233333,0.980649,1,0.549342
3031,3033,3041,-1,-0.044833,0.025750,1.0,0,0.362500,1.523509,1,0.549342


In [37]:
events.loc[train_df.index, 'primary_pred'] = train_df['primary_pred']
events.loc[train_df.index, 'primary_prob'] = train_df['primary_prob']


In [38]:
state_train = pd.read_parquet("data/module1_state_train.parquet")
state_val   = pd.read_parquet("data/module1_state_val.parquet")
state_test  = pd.read_parquet("data/module1_state_test.parquet")

state_df = pd.concat([state_train, state_val, state_test])
state_df = state_df.sort_values('ts').reset_index(drop=True)

events['t0_ts'] = df.loc[events['t0'], 'ts'].values
events['t1_ts'] = df.loc[events['t1'], 'ts'].values


In [39]:
meta_df = events.merge(
    state_df,
    left_on='t0_ts',
    right_on='ts',
    how='left'
)


In [40]:
meta_df.columns


Index(['t0', 't1', 'label', 'ret', 'sigma', 'primary_signal', 'meta_label',
       'uniqueness', 'sample_weight', 'primary_pred', 'primary_prob', 't0_ts',
       't1_ts', 'ts', 'p0', 'p1', 'p2', 'p3', 'p4', 'vol_pred', 'vol_regime',
       'trend_state'],
      dtype='object')

In [41]:
train_df.index.equals(events.loc[train_df.index].index)


True

In [42]:
meta_features = [
    'primary_pred',
    'primary_prob',
    'p0','p1','p2','p3','p4',
    'vol_pred'
]

X_meta = meta_df[meta_features].dropna()
y_meta = meta_df.loc[X_meta.index, 'meta_label']
w_meta = meta_df.loc[X_meta.index, 'sample_weight']


In [43]:
X_meta.isna().sum().sum()   # must be 0
# y_meta.isna().sum()         # must be 0
# w_meta.isna().sum()         # must be 0


np.int64(0)

In [44]:
y_meta.value_counts(normalize=True)


meta_label
1    0.521
0    0.479
Name: proportion, dtype: float64

In [45]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)


In [46]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)


In [47]:
from sklearn.metrics import precision_score, recall_score

precisions = []
signal_rates = []

for train_idx, test_idx in tscv.split(X_meta):
    X_tr, X_te = X_meta.iloc[train_idx], X_meta.iloc[test_idx]
    y_tr, y_te = y_meta.iloc[train_idx], y_meta.iloc[test_idx]
    w_tr = w_meta.iloc[train_idx]

    meta_model.fit(X_tr, y_tr, sample_weight=w_tr)

    preds = meta_model.predict(X_te)

    precisions.append(
        precision_score(y_te, preds, zero_division=0)
    )
    signal_rates.append(preds.mean())


In [48]:
print({
    "meta_precision": sum(precisions)/len(precisions),
    "meta_signal_rate": sum(signal_rates)/len(signal_rates)
})


{'meta_precision': 0.5140212488974658, 'meta_signal_rate': np.float64(0.44959999999999994)}


In [57]:
meta_df['t0_ts']

0      2017-09-08
1      2017-09-09
2      2017-09-10
3      2017-09-11
4      2017-09-12
          ...    
3010   2025-12-05
3011   2025-12-06
3012   2025-12-07
3013   2025-12-08
3014   2025-12-09
Name: t0_ts, Length: 3015, dtype: datetime64[ns]

In [58]:
meta_df.to_parquet(
    "data/btc_events.parquet",
    engine="pyarrow",
    index=False
)


In [56]:
meta_df['t1_ts']

0      2017-09-13
1      2017-09-13
2      2017-09-14
3      2017-09-13
4      2017-09-13
          ...    
3010   2025-12-09
3011   2025-12-09
3012   2025-12-15
3013   2025-12-14
3014   2025-12-12
Name: t1_ts, Length: 3015, dtype: datetime64[ns]